## Create the GUI

In [1]:
import pybullet as p

pybullet build time: Nov 28 2023 23:51:11


In [2]:
p.connect(p.GUI)
# without GUI: pybullet.connect(pybullet.DIRECT)

0

In [3]:
p.resetSimulation()

In [4]:
import pybullet_data
p.setAdditionalSearchPath(pybullet_data.getDataPath())

In [5]:
plane = p.loadURDF("plane.urdf")

In [6]:
for joint_index in range(p.getNumJoints(plane)):
    # Get dynamics info for the joint
    dynamics_info = p.getDynamicsInfo(plane, joint_index)

    # Modify the mass (and other parameters if needed)
    p.changeDynamics(plane, joint_index, lateralFriction=1.0,)

In [7]:
p.getDynamicsInfo(plane, -1)

(0.0,
 1.0,
 (0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 1.0),
 0.0,
 0.0,
 0.0,
 -1.0,
 -1.0,
 2,
 0.001)

## Additional extra URDF files

In [8]:
## this directory has lots of URDF files: maybe they might come in handy in future
pybullet_data.getDataPath()

'/home/sakib/.local/lib/python3.8/site-packages/pybullet_data'

In [9]:
!ls /home/z8/.local/lib/python3.8/site-packages/pybullet_data

ls: cannot access '/home/z8/.local/lib/python3.8/site-packages/pybullet_data': No such file or directory


In [10]:
!ls /home/z8/.local/lib/python3.8/site-packages/pybullet_data/table

ls: cannot access '/home/z8/.local/lib/python3.8/site-packages/pybullet_data/table': No such file or directory


## Import a table first

In [11]:
table = p.loadURDF('table/table.urdf', globalScaling=2.0)

In [12]:
p.getDynamicsInfo(table, -1)

(0.0,
 1.0,
 (0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 1.0),
 0.0,
 0.0,
 0.0,
 -1.0,
 -1.0,
 2,
 0.001)

In [13]:
for joint_index in range(p.getNumJoints(table)):
    # Get dynamics info for the joint
    dynamics_info = p.getDynamicsInfo(table, joint_index)

    # Modify the mass (and other parameters if needed)
    p.changeDynamics(table, joint_index, mass=20, lateralFriction=1.0)


In [14]:
# import time
# for _ in range(2000):
#     p.stepSimulation()
#     time.sleep(1.0 / 240)  # 240 Hz simulation, adjust as needed

In [15]:
import numpy as np
boundaries = p.getAABB(table)
lwh = np.array(boundaries[1])-np.array(boundaries[0])
print(boundaries)
print(lwh)

((-1.501, -1.001, 1.149), (1.501, 1.001, 1.251))
[3.002 2.002 0.102]


## Import Kinova j2s6s300 URDF file

In [17]:
robotic_arm = p.loadURDF('/home/sakib/catkin_ws/src/kinova-ros/kinova_description/urdf/j2s6s300_urdf.urdf', basePosition=[1.4,0,1.251], baseOrientation=[0,0,0,1], useFixedBase=True, globalScaling=1.0)

In [18]:
## experiment with the joints
## Now we can explore the world a little bit in numbers. For example, we can request the position and orientation of the robot in the world.

position, orientation = p.getBasePositionAndOrientation(robotic_arm)
orientation

(0.0, 0.0, 0.0, 1.0)

In [19]:
p.getDynamicsInfo(robotic_arm, -1) ## right now the mass is zero. Change it.

(0.0,
 0.5,
 (0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0),
 (0.0, 0.0, 0.0, 1.0),
 0.0,
 0.0,
 0.0,
 -1.0,
 -1.0,
 2,
 0.001)

In [20]:
for joint_index in range(p.getNumJoints(robotic_arm)):
    # Get dynamics info for the joint
    dynamics_info = p.getDynamicsInfo(robotic_arm, joint_index)

    # Modify the mass (and other parameters if needed)
    p.changeDynamics(robotic_arm, joint_index, mass=2)

## Importing ball

In [ ]:
tennis_ball = p.loadURDF('sphere_small.urdf', basePosition=[0.5,0,1.351], baseOrientation=[0,0,0,1], useFixedBase=True, globalScaling=1.0)

boundaries = p.getAABB(tennis_ball)
lwh = np.array(boundaries[1])-np.array(boundaries[0])
print(boundaries)
print(lwh)

In [ ]:
p.changeDynamics(tennis_ball,-1,linearDamping=0, angularDamping=0, rollingFriction=0.001, spinningFriction=0.001, mass = 0.02)

## Experiment with joints

In [ ]:
# Orientation is usually given in quaternions (x, y, z, w).

# We can ask for the number of joints of the robot.

p.getNumJoints(robotic_arm)

In [ ]:
# We can request information about each joint.

for joint_index in range(15):
    joint_info = p.getJointInfo(robotic_arm, joint_index)
    name, joint_type, lower_limit, upper_limit = \
        joint_info[1], joint_info[2], joint_info[8], joint_info[9]
    print(joint_index, name, joint_type, lower_limit, upper_limit)

In [ ]:
# There are more information in the tuple returned by pybullet.getJointInfo(...).

# We could as well request the current state of each joint, for example, the positions.

joint_positions = [j[0] for j in p.getJointStates(robotic_arm, range(15))]
joint_positions

In [ ]:
# or we could ask for the current position of a link.

world_position, world_orientation = p.getLinkState(robotic_arm, 4)[:2]
world_position

## Setup Simulation

In [ ]:
# Let's set up the simulation:

p.setGravity(0, 0, -9.81)   # everything should fall down
# pybullet.setTimeStep(0.0001)       # this slows everything down, but let's be accurate...
p.setRealTimeSimulation(0)  # we want to be faster than real time :)

In [ ]:
# Let's give the robot something to do. We will set the desired joint angle. There are other control modes: velocity control and torque control.

p.setJointMotorControlArray(
    robotic_arm, range(8), p.POSITION_CONTROL,
    targetPositions=[0, 0.0, 3.1, 2.3, 3.3, 1.5, 2.1, 1.1])
# Now, we can step through the simulation:
import time
for _ in range(1000):
    p.stepSimulation()
    time.sleep(1.0 / 240)  # 240 Hz simulation, adjust as needed

## Throwing the ball

In [ ]:
p.changeDynamics(tennis_ball, -1, restitution=0.6)
p.changeDynamics(tennis_ball, -1, mass=0.01)

In [ ]:
force = [0.10, 0.1, 0]  # Adjust the force vector as needed


p.applyExternalForce(tennis_ball, -1, force, [0, 0, 0.6], p.LINK_FRAME)

import time
for _ in range(2000):
    p.stepSimulation()
    time.sleep(1.0 / 240)  # 240 Hz simulation, adjust as needed

In [ ]:
## velocity control
# maxForce = 500
# pybullet.setJointMotorControl2(bodyUniqueId=robotic_arm,
# linkIndex=2,
# controlMode=pybullet.VELOCITY_CONTROL,
# targetVelocity = 0.2,
# force = maxForce)

In [ ]:
# pybullet.disconnect()

In [ ]:
from pybullet_utils import bullet_client as bc
from pybullet_utils import urdfEditor as ed
import pybullet
import pybullet_data
import time

p0 = bc.BulletClient(connection_mode=pybullet.DIRECT)
p0.setAdditionalSearchPath(pybullet_data.getDataPath())

p1 = bc.BulletClient(connection_mode=pybullet.DIRECT)
p1.setAdditionalSearchPath(pybullet_data.getDataPath())

#can also connect using different modes, GUI, SHARED_MEMORY, TCP, UDP, SHARED_MEMORY_SERVER, GUI_SERVER

husky = p1.loadURDF("husky/husky.urdf", flags=p0.URDF_USE_IMPLICIT_CYLINDER)
kuka = p0.loadURDF("kuka_iiwa/model.urdf")

ed0 = ed.UrdfEditor()
ed0.initializeFromBulletBody(husky, p1._client)
ed1 = ed.UrdfEditor()
ed1.initializeFromBulletBody(kuka, p0._client)
#ed1.saveUrdf("combined.urdf")

parentLinkIndex = 0

jointPivotXYZInParent = [0, 0, 0]
jointPivotRPYInParent = [0, 0, 0]

jointPivotXYZInChild = [0, 0, 0]
jointPivotRPYInChild = [0, 0, 0]

newjoint = ed0.joinUrdf(ed1, parentLinkIndex, jointPivotXYZInParent, jointPivotRPYInParent,
                        jointPivotXYZInChild, jointPivotRPYInChild, p0._client, p1._client)
newjoint.joint_type = p0.JOINT_FIXED

ed0.saveUrdf("combined.urdf")

print(p0._client)
print(p1._client)
print("p0.getNumBodies()=", p0.getNumBodies())
print("p1.getNumBodies()=", p1.getNumBodies())

pgui = bc.BulletClient(connection_mode=pybullet.GUI)
pgui.configureDebugVisualizer(pgui.COV_ENABLE_RENDERING, 0)

orn = [0, 0, 0, 1]
ed0.createMultiBody([0, 0, 0], orn, pgui._client)
pgui.setRealTimeSimulation(1)

pgui.configureDebugVisualizer(pgui.COV_ENABLE_RENDERING, 1)

while (pgui.isConnected()):
  pgui.getCameraImage(320, 200, renderer=pgui.ER_BULLET_HARDWARE_OPENGL)
  time.sleep(1. / 240.)

pybullet build time: Feb  4 2024 12:55:26
2024-02-06 13:33:40.964 Python[25573:14478492] WARNING: Secure coding is automatically enabled for restorable state! However, not on all supported macOS versions of this application. Opt-in to secure coding explicitly by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState:.


argv[0]=
argv[0]=
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
base_footprintb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
imu_linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
top_plate_linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/Import